[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Convolutional Networks &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import contextlib
import os

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


MARKS = ["horizontal", "vertical", "square"]


def mark_images(n=900, seed=2025):
    """Made-up 12 by 12 pictures, each a noisy background with one mark somewhere on it:
    a horizontal line, a vertical line or the outline of a square."""
    rng = np.random.default_rng(seed)
    labels = rng.integers(0, 3, n)
    images = rng.normal(0.0, 0.25, (n, 12, 12))
    for image, label in zip(images, labels):
        row, col = rng.integers(1, 8, 2)
        if label == 0:
            image[row + 2, col:col + 5] += 1.0
        elif label == 1:
            image[row:row + 5, col + 2] += 1.0
        else:
            image[row:row + 4, col:col + 4] += 1.0
            image[row + 1:row + 3, col + 1:col + 3] -= 1.0
    return images.astype("float32"), labels


def draw(image, threshold=0.6):
    """Print a picture as text, # where a pixel is above the threshold."""
    for row in np.asarray(image):
        print("".join("#" if value > threshold else "." for value in row))


images, labels = mark_images()
train_rows, test_rows = np.arange(600), np.arange(600, 900)

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(images), "pictures")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 900 pictures


The worked examples' tensors and training loop, built from the same made-up pictures, rebuilt here so
that each answer runs on its own.

In [2]:
batch = torch.from_numpy(images[:4]).unsqueeze(1)
X_train = torch.from_numpy(images[train_rows]).unsqueeze(1)
y_train = torch.from_numpy(labels[train_rows])
X_test = torch.from_numpy(images[test_rows]).unsqueeze(1)
y_test = torch.from_numpy(labels[test_rows])


def train(model, epochs=20):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    loss_fn = torch.nn.CrossEntropyLoss()
    generator = torch.Generator().manual_seed(0)
    for epoch in range(1, epochs + 1):
        order = torch.randperm(len(X_train), generator=generator)
        for start in range(0, len(X_train), 32):
            rows = order[start:start + 32]
            optimizer.zero_grad()
            loss_fn(model(X_train[rows]), y_train[rows]).backward()
            optimizer.step()
        if epoch % 5 == 0:
            with torch.no_grad():
                train_accuracy = (model(X_train).argmax(dim=1) == y_train).float().mean().item()
                test_accuracy = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
            print(epoch, "training", round(train_accuracy, 3), "| test", round(test_accuracy, 3))

**1.** Print the shape that `nn.Conv2d(1, 4, 5)` gives for `batch`, the first four pictures, and the size the rule from The idea predicts for a 5 by 5 filter with no padding.

In [3]:
torch.manual_seed(0)
print(tuple(torch.nn.Conv2d(1, 4, 5)(batch).shape))
print((12 + 2 * 0 - 5) // 1 + 1)

(4, 4, 8, 8)
8


The layer makes four maps of 8 by 8 for each of the four pictures, and (12 − 5) ÷ 1 + 1 = 8.

**2.** Count the parameters of `nn.Conv2d(8, 16, 3)` with `numel`, and write the count as a product of the filters, the channels, the filter's size and the biases.

In [4]:
layer = torch.nn.Conv2d(8, 16, 3)
print(sum(p.numel() for p in layer.parameters()))
print(16 * 8 * 3 * 3 + 16)

1168
1168


The layer has 1,168 parameters. Every one of its 16 filters has 3 × 3 weights for each of the 8 channels it reads, and a bias.

**3.** Write a filter that responds to horizontal lines, apply it to the first horizontal picture, `images[labels == 0][0]`, and `draw` its response with a threshold of 3.

In [5]:
horizontal_filter = torch.tensor([[[[-1.0, -1.0, -1.0],
                                    [2.0, 2.0, 2.0],
                                    [-1.0, -1.0, -1.0]]]])
picture = torch.from_numpy(images[labels == 0][0])
draw(picture)
print()
draw(torch.nn.functional.conv2d(picture[None, None], horizontal_filter)[0, 0], threshold=3.0)

............
............
............
............
............
............
............
............
..#####.....
............
............
............

..........
..........
..........
..........
..........
..........
..........
..####....
..........
..........


The filter is the vertical one turned on its side. Its response marks the line one row higher than the picture shows it, because a map made without padding starts with the filter centered on the picture's second row and column.

**4.** Train the PyTorch network with 4 and 8 filters in place of 8 and 16, from `torch.manual_seed(0)` with `train`, and print its parameter count.

In [6]:
torch.manual_seed(0)
small = torch.nn.Sequential(
    torch.nn.Conv2d(1, 4, 3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Conv2d(4, 8, 3, padding=1),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(8 * 3 * 3, 3),
)
print(sum(p.numel() for p in small.parameters()), "parameters")
train(small)

555 parameters
5 training 0.967 | test 0.937
10 training 1.0 | test 0.983
15 training 1.0 | test 0.98
20 training 1.0 | test 0.987


The smaller network has a third of the parameters, 555. In this run, from one seed, its test accuracy after 20 epochs is 0.987 against the larger network's 0.993, a difference of two pictures, and it had 0.937 at the fifth epoch where the larger one had 0.993. One run is not enough to rank the two.

**5.** Build the Keras network with `padding="valid"` in both convolutions, print each layer's output shape, and say why the last `Dense` layer now has fewer weights.

In [7]:
keras.utils.set_random_seed(0)
valid = keras.Sequential([
    keras.Input((12, 12, 1)),
    keras.layers.Conv2D(8, 3, padding="valid", activation="relu"),
    keras.layers.MaxPooling2D(2),
    keras.layers.Conv2D(16, 3, padding="valid", activation="relu"),
    keras.layers.MaxPooling2D(2),
    keras.layers.Flatten(),
    keras.layers.Dense(3),
])
for layer in valid.layers:
    print(f"{type(layer).__name__:13}", layer.output.shape)
print(valid.layers[-1].kernel.shape, valid.count_params(), "parameters")

Conv2D        (None, 10, 10, 8)
MaxPooling2D  (None, 5, 5, 8)
Conv2D        (None, 3, 3, 16)
MaxPooling2D  (None, 1, 1, 16)
Flatten       (None, 16)
Dense         (None, 3)
(16, 3) 1299 parameters


Without padding, each convolution makes its maps two pixels smaller, 12 to 10 and 5 to 3, and pooling 3 by 3 maps leaves 1 by 1. `Flatten` then hands on 16 numbers, so the last layer's kernel is (16, 3), where with padding it was (144, 3).

---

&#8592; **Back to:** [Convolutional Networks](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/12-convolutional-networks.ipynb)  &nbsp;&middot;&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)
